Here also the results wre stored and saved on google drive

In [1]:
import sys
!git clone https://github.com/Ignas12345/masters_project_helper_functions.git
sys.path.append('/content/masters_project_helper_functions')

Cloning into 'masters_project_helper_functions'...
remote: Enumerating objects: 196, done.
remote: Counting objects: 100% (27/27), done.
remote: Compressing objects: 100% (18/18), done.
remote: Total 196 (delta 15), reused 20 (delta 9), pack-reused 169 (from 1)
Receiving objects: 100% (196/196), 75.70 KiB | 1.68 MiB/s, done.
Resolving deltas: 100% (114/114), done.


In [ ]:
mount_drive = 1
if mount_drive:
    from google.colab import drive
    drive.mount('/content/Gdrive')

Drive already mounted at /content/Gdrive; to attempt to forcibly remount, call drive.mount("/content/Gdrive", force_remount=True).


In [ ]:
import pandas as pd
from IPython.display import display
import ast
from sklearn import base

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC, LinearSVC
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, ExtraTreesClassifier
from sklearn.calibration import CalibratedClassifierCV

import masters_project_helper_functions.utils as utils
import masters_project_helper_functions.plotting as plotting
import masters_project_helper_functions.preprocessing_methods as pp
import masters_project_helper_functions.feature_selection as feature_selection
import masters_project_helper_functions.classification_pipeline as classification_pipeline

In [ ]:
url_TGCT_non_TGCT_mirna_rpm_counts = "https://raw.githubusercontent.com/Ignas12345/masters_project_data_and_notebooks/refs/heads/main/Data/expression_values/TGCT_non_TGCT_GTEx_combined_mirna_rpm_counts.csv"
sample_label_dict = pd.read_csv('https://raw.githubusercontent.com/Ignas12345/masters_project_data_and_notebooks/refs/heads/main/Data/sample_annotations/TGCT_vs_non_TGCT_labels.csv', sep = ';', index_col=0)['label'].to_dict()

df = pd.read_csv(url_TGCT_non_TGCT_mirna_rpm_counts, sep =';', index_col = 0)
results_df = pd.read_csv('/content/Gdrive/MyDrive/Magistro_projektas/loocv_results/TGCT_vs_non_TGCT_results_df.csv', sep = ';', index_col = 0)

In [ ]:
experiment_name = 'TGCT_vs_non_TGCT'
fold_path = 'https://raw.githubusercontent.com/Ignas12345/masters_project_data_and_notebooks/refs/heads/main/Data/bootstrap_folds/'
'''

info_df = pd.read_csv(fold_path + 'info_df_' + f'{experiment_name}.csv', index_col = 0)
train_folds_df = pd.read_csv(fold_path + 'train_folds_' + f'{experiment_name}.csv', index_col = 0)
test_folds_df = pd.read_csv(fold_path + 'test_folds_' + f'{experiment_name}.csv', index_col = 0)
group_labels = [label for label in list(set(sample_label_dict.values())) if label != 'unused']

loocv_fold_indices = utils.get_folds_by_comment(train_folds_df, 'loocv fold')
'''

"\n\ninfo_df = pd.read_csv(fold_path + 'info_df_' + f'{experiment_name}.csv', index_col = 0)\ntrain_folds_df = pd.read_csv(fold_path + 'train_folds_' + f'{experiment_name}.csv', index_col = 0)\ntest_folds_df = pd.read_csv(fold_path + 'test_folds_' + f'{experiment_name}.csv', index_col = 0)\ngroup_labels = [label for label in list(set(sample_label_dict.values())) if label != 'unused']\n\nloocv_fold_indices = utils.get_folds_by_comment(train_folds_df, 'loocv fold')\n"

In [ ]:
pre_processing_methods = {'initial_feature_filtering' : pp.feature_filtering_by_class_means,
                          'sample_wise_scaling' : pp.normalize_by_housekeeping_list,
                          'feature_wise_scaling' : pp.log_normalization,
                          'rank_features_and_keep_top_n' : pp.rank_features_and_keep_top_n_features,
                          }
feature_selection_method = feature_selection.rfecv_feature_selection

feature_ranking_method = pp.perform_RFE_ranking
kwargs = {'housekeeping_list' : ['hsa-mir-191, mature,MIMAT0000440', ],
          'scale_housekeep_by_mean' : False,
          'feature_ranking_method' : feature_ranking_method,
          'keep_n_ranked_features' : 10,
          #'drop_correlated_features': True
          }

'''
kwargs['smaller_group_label'] = min(group_labels, key=lambda x: len([sample for sample, label in sample_label_dict.items() if label == x]))
kwargs['larger_group_label'] = max(group_labels, key=lambda x: len([sample for sample, label in sample_label_dict.items() if label == x]))
'''

classification_method = LogisticRegression
possible_parameters = [{'C' : 10, 'class_weight' : 'balanced', 'penalty' : 'l1', 'solver' : 'liblinear', 'max_iter' : 1000}]

kwargs['classification_method_parameters'] = possible_parameters[0]

In [ ]:
result, detailed_result = classification_pipeline.run_pipeline_on_loocv_folds_and_record_performance(df, fold_path, experiment_name, sample_label_dict, pre_processing_methods, feature_selection_method, classification_method, **kwargs)

In [ ]:
results_df.to_csv('/content/Gdrive/MyDrive/Magistro_projektas/loocv_results/TGCT_vs_non_TGCT_results_df_back_up.csv', sep = ';')
results_df = pd.concat([results_df, result]).reset_index(drop=True)
results_df.to_csv('/content/Gdrive/MyDrive/Magistro_projektas/loocv_results/TGCT_vs_non_TGCT_results_df.csv', sep = ';')
display(results_df)

,experiment_name,initial_feature_filtering,sample_wise_scaling,rank_features_and_keep_top_n,feature_wise_scaling,feature_selection_method,classification_method,housekeeping_list,scale_housekeep_by_mean,feature_ranking_method,...,mean_feature_number,balanced_accuracy_score,recall_score,precision_score,brier_score_loss,matthews_corrcoef,classification_method_parameters,keep_n_ranked_features,estimator_for_rfe_ranking,time_taken
0,TGCT_vs_non_TGCT,feature_filtering_by_class_means,normalize_by_housekeeping_list,rank_features_and_keep_top_n_features,log_normalization,select_all_features,SVC,"['hsa-mir-191, mature,MIMAT0000440']",False,perform_DE_ranking,...,231.858268,1.0,1.0,1.0,0.031896,1.0,NaN,NaN,NaN,NaN
1,TGCT_vs_non_TGCT,feature_filtering_by_class_means,normalize_by_housekeeping_list,rank_features_and_keep_top_n_features,log_normalization,select_all_features,LinearSVC,"['hsa-mir-191, mature,MIMAT0000440']",False,perform_DE_ranking,...,230.858268,1.0,1.0,1.0,0.075036,1.0,"{'C': 1000, 'class_weight': 'balanced'}",NaN,NaN,NaN
2,TGCT_vs_non_TGCT,feature_filtering_by_class_means,normalize_by_housekeeping_list,rank_features_and_keep_top_n_features,log_normalization,rfecv_feature_selection,SVC,"['hsa-mir-191, mature,MIMAT0000440']",False,perform_RFE_ranking,...,7.358268,0.993261,1.0,0.964789,0.005397,0.975595,"{'C': 1000, 'class_weight': 'balanced', 'proba...",10.0,NaN,NaN
3,TGCT_vs_non_TGCT,feature_filtering_by_class_means,normalize_by_housekeeping_list,rank_features_and_keep_top_n_features,log_normalization,rfecv_feature_selection,SVC,"['hsa-mir-191, mature,MIMAT0000440']",False,perform_RFE_ranking,...,3.417323,0.993655,0.992701,0.985507,0.009428,0.985055,"{'C': 1000, 'class_weight': 'balanced', 'proba...",5.0,"SVC(C=1000, class_weight='balanced', kernel='l...",NaN
4,TGCT_vs_non_TGCT,feature_filtering_by_class_means,normalize_by_housekeeping_list,rank_features_and_keep_top_n_features,log_normalization,rfecv_feature_selection,LogisticRegression,"['hsa-mir-191, mature,MIMAT0000440']",False,perform_RFE_ranking,...,6.897638,0.997305,1.0,0.985612,0.004178,0.9901,"{'C': 100, 'class_weight': 'balanced'}",10.0,"LogisticRegression(C=100, class_weight='balanc...",NaN
5,TGCT_vs_non_TGCT,feature_filtering_by_class_means,normalize_by_housekeeping_list,rank_features_and_keep_top_n_features,log_normalization,rfecv_feature_selection,ExtraTreesClassifier,"['hsa-mir-191, mature,MIMAT0000440']",False,perform_RFE_ranking,...,6.244094,0.995003,0.992701,0.992701,0.002387,0.990005,NaN,10.0,ExtraTreesClassifier(),NaN
6,TGCT_vs_non_TGCT,feature_filtering_by_class_means,normalize_by_housekeeping_list,rank_features_and_keep_top_n_features,log_normalization,rfecv_feature_selection,LogisticRegression,"['hsa-mir-191, mature,MIMAT0000440']",False,perform_RFE_ranking,...,6.846457,0.994609,1.0,0.971631,0.0051,0.980385,"{'C': 1, 'class_weight': 'balanced'}",10.0,"LogisticRegression(C=1, class_weight='balanced')",2851.26041
7,TGCT_vs_non_TGCT,feature_filtering_by_class_means,normalize_by_housekeeping_list,rank_features_and_keep_top_n_features,log_normalization,rfecv_feature_selection,LogisticRegression,"['hsa-mir-191, mature,MIMAT0000440']",False,perform_RFE_ranking,...,4.021654,0.998652,1.0,0.992754,0.002813,0.995027,"{'C': 100, 'class_weight': 'balanced'}",5.0,"LogisticRegression(C=100, class_weight='balanc...",2135.666985
8,TGCT_vs_non_TGCT,feature_filtering_by_class_means,normalize_by_housekeeping_list,rank_features_and_keep_top_n_features,log_normalization,rfecv_feature_selection,LogisticRegression,"['hsa-mir-191, mature,MIMAT0000440']",False,perform_RFE_ranking,...,4.021654,0.998652,1.0,0.992754,0.002813,0.995027,"{'C': 100, 'class_weight': 'balanced'}",5.0,"LogisticRegression(C=100, class_weight='balanc...",2134.687293
9,TGCT_vs_non_TGCT,feature_filtering_by_class_means,normalize_by_housekeeping_list,rank_features_and_keep_top_n_features,log_normalization,rfecv_feature_selection,LogisticRegression,"['hsa-mir-191, mature,MIMAT0000440']",False,perform_RFE_rank

In [ ]:
classification_method = SVC
possible_parameters = [{'C' : 1000, 'kernel' : 'rbf', 'probability' : True, 'class_weight' : 'balanced'}, {'C' : 1000, 'kernel' : 'linear', 'probability' : True, 'class_weight' : 'balanced'},
                       {'C' : 10, 'kernel' : 'rbf', 'probability' : True, 'class_weight' : 'balanced'}, {'C' : 10, 'kernel' : 'linear', 'probability' : True, 'class_weight' : 'balanced'}]

kwargs['classification_method_parameters'] = possible_parameters[0]

In [ ]:
classification_method = SVC
possible_parameters = [{'C' : 1000, 'class_weight' : 'balanced', 'probability' : True, 'kernel' : 'linear'}]

kwargs['classification_method_parameters'] = possible_parameters[0]

In [ ]:
result, detailed_result = classification_pipeline.run_pipeline_on_loocv_folds_and_record_performance(df, fold_path, experiment_name, sample_label_dict, pre_processing_methods, feature_selection_method, classification_method, **kwargs)

In [ ]:
display(detailed_result)
result

,prediction,prob_class_1,true label,number_of_features_used
TCGA-2G-AAEW-01,1,1.0,1,4.0
TCGA-2G-AAEX-01,1,1.0,1,4.0
TCGA-2G-AAF1-01,1,1.0,1,4.0
TCGA-2G-AAF4-01,1,1.0,1,4.0
TCGA-2G-AAF6-01,1,1.0,1,4.0
...,...,...,...,...
GTEX-ZVTK-0126-SM-EV6TS,0,0.000662,0,6.0
GTEX-ZYT6-2726-SM-EVLK1,0,0.000791,0,6.0
GTEX-ZZ64-1126-SM-DECQM,0,0.000196,0,6.0
1_TCGA-TQ-A7RK-02,0,0.001152,0,6.0


,experiment_name,initial_feature_filtering,sample_wise_scaling,feature_wise_scaling,rank_features_and_keep_top_n,feature_selection_method,classification_method,housekeeping_list,scale_housekeep_by_mean,feature_ranking_method,...,smaller_group_label,larger_group_label,classifier_for_feature_selection,accuracy_score,mean_feature_number,balanced_accuracy_score,recall_score,precision_score,brier_score_loss,matthews_corrcoef
0,TGCT_vs_non_TGCT,feature_filtering_by_class_means,normalize_by_housekeeping_list,log_normalization,rank_features_and_keep_top_n_features,rfecv_feature_selection,SVC,"[hsa-mir-191, mature,MIMAT0000440]",False,perform_RFE_ranking,...,TGCT,non_TGCT,"SVC(C=1000, class_weight='balanced', kernel='l...",0.990157,7.358268,0.993261,1.0,0.964789,0.005397,0.975595
